In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

# 1. 간단한 toy 단어장(영어-불어)과 토크나이저 함수 정의
SRC_VOCAB = {"<pad>":0, "<sos>":1, "<eos>":2, "hello":3, "world":4}   # 영어 단어와 특수토큰
TRG_VOCAB = {"<pad>":0, "<sos>":1, "<eos>":2, "bonjour":3, "le":4, "monde":5}  # 불어 단어와 특수토큰
SRC_itos = {v:k for k,v in SRC_VOCAB.items()}   # 숫자 인덱스 → 단어
TRG_itos = {v:k for k,v in TRG_VOCAB.items()}   # 숫자 인덱스 → 단어

def tokenize_src(sentence):
    # 영어 문장 → [<sos>, 단어 인덱스들, <eos>] 리스트로 변환
    return [SRC_VOCAB["<sos>"]] + [SRC_VOCAB[w] for w in sentence.split()] + [SRC_VOCAB["<eos>"]]

def tokenize_trg(sentence):
    # 불어 문장 → [<sos>, 단어 인덱스들, <eos>] 리스트로 변환
    return [TRG_VOCAB["<sos>"]] + [TRG_VOCAB[w] for w in sentence.split()] + [TRG_VOCAB["<eos>"]]

# 2. 데이터 (배치 크기 1, 하나의 문장만 예시)
src_sentences = ["hello world"]         # 입력(영어)
trg_sentences = ["bonjour le monde"]    # 출력(불어)

# 각 문장을 숫자 인덱스 시퀀스로 변환하고, (seq_len, batch) 형태로 변환
src_tensor = torch.tensor([tokenize_src(s) for s in src_sentences]).T
trg_tensor = torch.tensor([tokenize_trg(s) for s in trg_sentences]).T

# 3. Encoder 정의
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, hidden_dim)        # 입력 단어를 임베딩 벡터로 변환
        self.rnn = nn.LSTM(hidden_dim, hidden_dim, num_layers)      # LSTM RNN 계층 정의

    def forward(self, x):
        embedded = self.embedding(x)                                # (seq_len, batch) → (seq_len, batch, hidden_dim)
        outputs, (hidden, cell) = self.rnn(embedded)                # LSTM 통과, hidden/cell은 마지막 시점의 상태
        return hidden, cell                                         # 디코더로 전달할 hidden, cell 반환

# 4. Decoder 정의
class Decoder(nn.Module):
    def __init__(self, output_dim, hidden_dim, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, hidden_dim)       # 출력 단어를 임베딩 벡터로 변환
        self.rnn = nn.LSTM(hidden_dim, hidden_dim, num_layers)      # LSTM RNN 계층 정의
        self.fc = nn.Linear(hidden_dim, output_dim)                 # LSTM 출력 → 단어 확률분포 변환하는 완전연결층

    def forward(self, x, hidden, cell):
        embedded = self.embedding(x.unsqueeze(0))                   # 입력(토큰)을 임베딩, shape=(1, batch, hidden_dim)
        outputs, (hidden, cell) = self.rnn(embedded, (hidden, cell))# 이전 hidden/cell로부터 LSTM 한스텝 실행
        prediction = self.fc(outputs.squeeze(0))                    # (batch, hidden_dim) → (batch, output_dim) 단어 분포
        return prediction, hidden, cell                             # 다음 단어 분포, hidden, cell 반환

# 5. Seq2Seq 전체 모델 정의 (encoder, decoder 합침)
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder      # Encoder 객체 저장
        self.decoder = decoder      # Decoder 객체 저장

    def forward(self, source, target, teacher_forcing_ratio=0.5):
        batch_size = source.shape[1]                          # 배치 크기 (여기선 1)
        target_len = target.shape[0]                          # 출력 시퀀스 길이
        outputs = torch.zeros(target_len, batch_size, self.decoder.fc.out_features)  # 결과 저장용 텐서

        hidden, cell = self.encoder(source)                   # 입력 시퀀스를 인코더에 넣어 마지막 hidden/cell 얻음
        decoder_input = target[0, :]                          # 디코더 첫 입력(<sos> 토큰)

        for t in range(1, target_len):                        # 각 출력 시점 반복 (1부터 시작, <sos>는 제외)
            output, hidden, cell = self.decoder(decoder_input, hidden, cell)   # 디코더 한 스텝 실행
            outputs[t] = output                               # 예측 결과 저장
            teacher_force = random.random() < teacher_forcing_ratio            # teacher forcing 적용 여부
            decoder_input = target[t] if teacher_force else output.argmax(1)   # 정답 or 예측 결과로 다음 입력 결정

        return outputs                                        # 전체 출력(softmax logits) 반환

# 6. 하이퍼파라미터, 모델, 손실함수, 옵티마이저 정의
INPUT_DIM = len(SRC_VOCAB)       # 입력 단어장 크기
OUTPUT_DIM = len(TRG_VOCAB)      # 출력 단어장 크기
HIDDEN_DIM = 32                  # 은닉층 크기
NUM_LAYERS = 1                   # LSTM 레이어 개수
PAD_IDX = 0                      # 패딩 토큰 인덱스

encoder = Encoder(INPUT_DIM, HIDDEN_DIM, NUM_LAYERS)          # 인코더 객체
decoder = Decoder(OUTPUT_DIM, HIDDEN_DIM, NUM_LAYERS)         # 디코더 객체
model = Seq2Seq(encoder, decoder)                             # Seq2Seq 전체 모델

optimizer = optim.Adam(model.parameters())                    # Adam 옵티마이저
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)         # 패딩 무시한 CrossEntropyLoss

# 7. 학습 루프 (100 에폭 반복)
EPOCHS = 100
for epoch in range(EPOCHS):
    model.train()                                             # 학습 모드
    optimizer.zero_grad()                                     # 이전 그래디언트 초기화
    output = model(src_tensor, trg_tensor)                    # 모델 순전파 (output: [target_len, batch, output_dim])
    output_dim = output.shape[-1]                             # 출력 단어장 크기
    output_flat = output[1:].view(-1, output_dim)             # <sos> 제외, (target_len-1, batch, output_dim) → ((target_len-1)*batch, output_dim)
    trg_flat = trg_tensor[1:].reshape(-1)                     # <sos> 제외, (target_len-1, batch) → ((target_len-1)*batch)
    loss = criterion(output_flat, trg_flat)                   # loss 계산
    loss.backward()                                           # 역전파
    optimizer.step()                                          # 가중치 갱신
    if (epoch+1) % 20 == 0 or epoch == 0:                     # 20 에폭마다 loss 출력
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# 8. 번역(추론) 함수 정의
def translate(model, src_tensor, max_len=10):
    model.eval()                                              # 평가 모드
    with torch.no_grad():                                     # 그래디언트 비활성화
        hidden, cell = model.encoder(src_tensor)              # 입력 문장을 인코더에 넣어 hidden, cell 얻음
        input_token = torch.tensor([TRG_VOCAB["<sos>"]])      # 첫 디코더 입력: <sos>
        outputs = []                                          # 생성 결과 저장 리스트
        for _ in range(max_len):                              # 최대 max_len까지 반복
            output, hidden, cell = model.decoder(input_token, hidden, cell)    # 디코더 한 스텝
            pred_token = output.argmax(1).item()              # 예측 단어(가장 높은 확률의 인덱스)
            if pred_token == TRG_VOCAB["<eos>"]:              # <eos> 나오면 종료
                break
            outputs.append(pred_token)                        # 예측 단어 저장
            input_token = torch.tensor([pred_token])          # 다음 입력은 방금 예측 단어
    return [TRG_itos[idx] for idx in outputs]

# 9. 테스트 (hello world → bonjour le monde)
# 입력 문장(인덱스 → 단어)
input_tokens = [SRC_itos[int(idx)] for idx in src_tensor.squeeze()]
print("입력 문장:", " ".join(input_tokens))


# 번역 결과(이미 코드에 있음)
result = translate(model, src_tensor)
print("출력 문장:", " ".join(result))




Epoch 1, Loss: 1.7372
Epoch 20, Loss: 1.3525
Epoch 40, Loss: 0.8685
Epoch 60, Loss: 0.4645
Epoch 80, Loss: 0.2377
Epoch 100, Loss: 0.1369
입력 문장: <sos> hello world <eos>
출력 문장: bonjour le monde


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import random

# 1. Toy 단어장 & 토크나이저 함수 정의
SRC_VOCAB = {"<pad>":0, "<sos>":1, "<eos>":2, "hello":3, "world":4}   # 영어 단어 집합
TRG_VOCAB = {"<pad>":0, "<sos>":1, "<eos>":2, "bonjour":3, "le":4, "monde":5}   # 불어 단어 집합
SRC_itos = {v: k for k, v in SRC_VOCAB.items()}   # 인덱스 → 단어
TRG_itos = {v: k for k, v in TRG_VOCAB.items()}   # 인덱스 → 단어

def tokenize_src(sentence):
    # 영어 문장 → [<sos>, 단어 인덱스들, <eos>] 리스트로 변환
    return [SRC_VOCAB["<sos>"]] + [SRC_VOCAB[w] for w in sentence.split()] + [SRC_VOCAB["<eos>"]]

def tokenize_trg(sentence):
    # 불어 문장 → [<sos>, 단어 인덱스들, <eos>] 리스트로 변환
    return [TRG_VOCAB["<sos>"]] + [TRG_VOCAB[w] for w in sentence.split()] + [TRG_VOCAB["<eos>"]]

src_sentences = ["hello world"]         # 입력 문장(영어)
trg_sentences = ["bonjour le monde"]    # 출력 문장(불어)
src_tensor = torch.tensor([tokenize_src(s) for s in src_sentences]).T   # (seq_len, batch)
trg_tensor = torch.tensor([tokenize_trg(s) for s in trg_sentences]).T   # (seq_len, batch)

# 2. Bahdanau(Additive) Attention 모듈
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 2, hidden_dim)     # [hidden; hidden] concat 후 변환
        self.v = nn.Parameter(torch.rand(hidden_dim))         # attention 가중치 벡터(v_a)

    def forward(self, hidden, encoder_outputs):
        # hidden: [batch, hidden_dim]
        # encoder_outputs: [seq_len, batch, hidden_dim]
        seq_len = encoder_outputs.shape[0]
        batch_size = encoder_outputs.shape[1]
        # 디코더 hidden을 seq_len만큼 복제 → [seq_len, batch, hidden_dim]
        hidden = hidden.unsqueeze(0).repeat(seq_len, 1, 1)
        # [hidden, encoder_outputs] concat 후 tanh 통과 → energy 계산
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))  # [seq_len, batch, hidden_dim]
        # 배치가 앞에 오도록 transpose
        energy = energy.transpose(0, 1)  # [batch, seq_len, hidden_dim]
        # attention 벡터 v를 batch에 맞춰 확장
        v = self.v.repeat(batch_size, 1).unsqueeze(1)  # [batch, 1, hidden_dim]
        # v와 energy 내적하여 attention score 산출
        attention_scores = torch.bmm(v, energy.transpose(1, 2)).squeeze(1)  # [batch, seq_len]
        # score를 softmax로 정규화 → attention weight
        attention_weights = F.softmax(attention_scores, dim=1)  # [batch, seq_len]
        # attention weight로 encoder_outputs weighted sum → context vector
        context = torch.bmm(attention_weights.unsqueeze(1), encoder_outputs.transpose(0,1)).squeeze(1)  # [batch, hidden_dim]
        return context, attention_weights  # context vector, attention 분포 반환

# 3. Encoder 정의 (일반 LSTM)
class Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, hidden_dim)      # 입력 단어 인덱스 → 임베딩
        self.rnn = nn.LSTM(hidden_dim, hidden_dim, num_layers)    # LSTM 계층

    def forward(self, x):
        embedded = self.embedding(x)                     # [seq_len, batch, hidden_dim]
        outputs, (hidden, cell) = self.rnn(embedded)     # outputs: 전체 시퀀스의 hidden state
        return outputs, hidden, cell                     # outputs: [seq_len, batch, hidden_dim]

# 4. Attention Decoder
class Decoder(nn.Module):
    def __init__(self, output_dim, hidden_dim, num_layers, attention):
        super().__init__()
        self.embedding = nn.Embedding(output_dim, hidden_dim)     # 출력 단어 인덱스 → 임베딩
        self.rnn = nn.LSTM(hidden_dim * 2, hidden_dim, num_layers) # context vector 포함 입력
        self.fc = nn.Linear(hidden_dim * 2, output_dim)           # [hidden; context] → 단어 분포
        self.attention = attention

    def forward(self, x, hidden, cell, encoder_outputs):
        # x: [batch]
        # hidden/cell: [num_layers, batch, hidden_dim]
        # encoder_outputs: [seq_len, batch, hidden_dim]
        x = x.unsqueeze(0)                                   # [1, batch]
        embedded = self.embedding(x)                         # [1, batch, hidden_dim]
        embedded = embedded.squeeze(0)                       # [batch, hidden_dim]
        # Attention 계산 (디코더 현재 hidden과 인코더 outputs 사용)
        context, attn_weights = self.attention(hidden[-1], encoder_outputs)  # context: [batch, hidden_dim]
        # decoder input: [임베딩, context] concat
        rnn_input = torch.cat((embedded, context), dim=1).unsqueeze(0)       # [1, batch, hidden_dim*2]
        outputs, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))        # LSTM 업데이트
        outputs = outputs.squeeze(0)                                         # [batch, hidden_dim]
        # [LSTM output, context vector] concat 후 FC layer로 단어 확률분포 예측
        output_cat = torch.cat((outputs, context), dim=1)                    # [batch, hidden_dim*2]
        prediction = self.fc(output_cat)                                     # [batch, output_dim]
        return prediction, hidden, cell, attn_weights                        # 다음 단어 분포, state, attention 반환

# 5. Seq2Seq 통합 모델
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder      # Encoder 객체
        self.decoder = decoder      # Decoder 객체
        self.device = device        # CPU/GPU

    def forward(self, source, target, teacher_forcing_ratio=0.5):
        batch_size = source.shape[1]                    # 배치 크기
        target_len = target.shape[0]                    # 출력 시퀀스 길이
        output_dim = self.decoder.fc.out_features       # 출력 단어장 크기
        outputs = torch.zeros(target_len, batch_size, output_dim).to(self.device)  # 결과 저장

        encoder_outputs, hidden, cell = self.encoder(source)        # 인코더에 입력 문장 통과
        decoder_input = target[0, :]                               # 디코더 첫 입력: <sos> 토큰

        for t in range(1, target_len):                             # 각 시점 반복 (<sos> 제외)
            output, hidden, cell, attn_weights = self.decoder(
                decoder_input, hidden, cell, encoder_outputs
            )   # 디코더 한 스텝
            outputs[t] = output                                   # 예측 결과 저장
            teacher_force = random.random() < teacher_forcing_ratio
            # teacher forcing: 정답 입력 or 예측 결과 입력
            decoder_input = target[t] if teacher_force else output.argmax(1)
        return outputs                                            # 전체 시퀀스 예측 결과

    def translate(self, src_tensor, max_len=10):
        # 단일 문장(batch=1) 추론 함수
        self.eval()
        with torch.no_grad():
            encoder_outputs, hidden, cell = self.encoder(src_tensor)
            input_token = torch.tensor([TRG_VOCAB["<sos>"]]).to(self.device)
            outputs = []
            for _ in range(max_len):
                output, hidden, cell, attn_weights = self.decoder(
                    input_token, hidden, cell, encoder_outputs
                )
                pred_token = output.argmax(1).item()         # 가장 확률 높은 단어 선택
                if pred_token == TRG_VOCAB["<eos>"]:
                    break
                outputs.append(pred_token)
                input_token = torch.tensor([pred_token]).to(self.device)
        return [TRG_itos[idx] for idx in outputs]            # 예측 인덱스를 단어로 변환해 반환

# 6. 하이퍼파라미터 및 모델 초기화
INPUT_DIM = len(SRC_VOCAB)           # 입력 단어장 크기
OUTPUT_DIM = len(TRG_VOCAB)          # 출력 단어장 크기
HIDDEN_DIM = 32                      # 은닉층 크기
NUM_LAYERS = 1                       # LSTM 레이어 개수
PAD_IDX = 0                          # 패딩 토큰 인덱스
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

attention = Attention(HIDDEN_DIM)                        # attention 모듈 생성
encoder = Encoder(INPUT_DIM, HIDDEN_DIM, NUM_LAYERS)     # 인코더 객체
decoder = Decoder(OUTPUT_DIM, HIDDEN_DIM, NUM_LAYERS, attention)   # 디코더 객체
model = Seq2Seq(encoder, decoder, device).to(device)     # Seq2Seq 전체 모델

optimizer = optim.Adam(model.parameters())               # Adam optimizer
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)    # 패딩 무시 CrossEntropyLoss

src_tensor = src_tensor.to(device)   # 입력 텐서를 device로 이동
trg_tensor = trg_tensor.to(device)   # 출력 텐서를 device로 이동

# 7. 학습 루프
EPOCHS = 100
for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()                                 # 그래디언트 초기화
    output = model(src_tensor, trg_tensor)                # 순전파
    output_dim = output.shape[-1]
    output_flat = output[1:].view(-1, output_dim)         # <sos> 제외, (target_len-1, batch, output_dim) -> ((target_len-1)*batch, output_dim)
    trg_flat = trg_tensor[1:].reshape(-1)                 # 정답(<sos> 제외)도 1차원으로
    loss = criterion(output_flat, trg_flat)               # loss 계산
    loss.backward()                                       # 역전파
    optimizer.step()                                      # 파라미터 갱신
    if (epoch + 1) % 20 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")  # 중간중간 loss 출력

# 8. 테스트: 입력 문장 → 번역 결과 출력
input_tokens = [SRC_itos[int(idx)] for idx in src_tensor.squeeze()]
print("입력 문장:", " ".join(input_tokens))
result = model.translate(src_tensor)
print("출력 문장:", " ".join(result))



Epoch 1, Loss: 1.7671
Epoch 20, Loss: 1.3718
Epoch 40, Loss: 0.8357
Epoch 60, Loss: 0.3878
Epoch 80, Loss: 0.1816
Epoch 100, Loss: 0.0927
입력 문장: <sos> hello world <eos>
출력 문장: bonjour le monde


In [3]:
import torch
import torch.nn as nn
import math

# ------------------ 위치 인코딩 ------------------
class PositionalEncoding(nn.Module):
    """
    시퀀스 내 단어 순서 정보를 인코딩하여 입력 임베딩에 더해줌
    """
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        # 위치별로 고정된 포지션 인코딩 행렬 생성
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # 입력 임베딩에 위치 정보(pe) 더해줌
        x = x + self.pe[:, :x.size(1)]
        return x

# ------------------ Scaled Dot-Product Attention(self-attention score) ------------------
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q, K, V: (batch, head, seq_len, d_k)
    mask: (batch, 1, 1, seq_len) or (batch, 1, seq_len, seq_len)
    """
    d_k = Q.size(-1)
    # Q와 K의 내적 후 sqrt(d_k)로 나눔 (스케일링)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        # 마스킹: 패딩 토큰 또는 미래 정보 가림 (디코더에서 사용)
        scores = scores.masked_fill(mask == 0, -1e9)
    # 소프트맥스: 각 토큰별 가중치
    attn = torch.softmax(scores, dim=-1)
    # 가중평균
    output = torch.matmul(attn, V)
    return output, attn

# ------------------ 멀티헤드 어텐션 ------------------
class MultiHeadAttention(nn.Module):
    """
    Q, K, V를 여러 head로 쪼개 각각 어텐션 연산 → 최종 concat
    """
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        # Q, K, V 선형 변환
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        self.W_O = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V, mask=None):
        batch_size = Q.size(0)

        # Q, K, V 변환 및 head 분할
        Q = self.W_Q(Q).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(K).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(V).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

        # Scaled Dot-Product Attention
        out, attn = scaled_dot_product_attention(Q, K, V, mask)

        # 여러 head 결과를 합쳐(d_model) 다시 projection
        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        out = self.W_O(out)
        return out

# ------------------ FFN ------------------
class PositionwiseFeedForward(nn.Module):
    """
    각 토큰별 완전연결(FFN, 비선형 ReLU) 두 층
    """
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

# ------------------ 인코더 블록 ------------------
class EncoderLayer(nn.Module):
    """
    인코더의 한 층: (1) Self-Attention (2) Add & Norm (3) FFN (4) Add & Norm
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionwiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # (1) Self-Attention & Add & Norm
        attn_out = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_out))
        # (2) FFN & Add & Norm
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x

# ------------------ 디코더 블록 ------------------
class DecoderLayer(nn.Module):
    """
    디코더의 한 층: (1) Masked Self-Attention (2) Add & Norm
                  (3) Encoder-Decoder Attention (4) Add & Norm
                  (5) FFN (6) Add & Norm
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.enc_dec_attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = PositionwiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, enc_out, tgt_mask=None, src_mask=None):
        # (1) Masked Self-Attention & Add & Norm
        attn_out = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_out))
        # (2) Encoder-Decoder Attention & Add & Norm
        attn_out = self.enc_dec_attn(x, enc_out, enc_out, src_mask)
        x = self.norm2(x + self.dropout(attn_out))
        # (3) FFN & Add & Norm
        ffn_out = self.ffn(x)
        x = self.norm3(x + self.dropout(ffn_out))
        return x

# ------------------ 전체 인코더 ------------------
class Encoder(nn.Module):
    """
    인코더 전체: 임베딩 → 포지셔널 인코딩 → N개 인코더 블록 반복
    """
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, max_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])

    def forward(self, src, src_mask=None):
        # (1) 임베딩 + 포지셔널 인코딩
        x = self.embedding(src)
        x = self.pos_encoding(x)
        # (2) N개 인코더 블록 반복
        for layer in self.layers:
            x = layer(x, src_mask)
        return x

# ------------------ 전체 디코더 ------------------
class Decoder(nn.Module):
    """
    디코더 전체: 임베딩 → 포지셔널 인코딩 → N개 디코더 블록 반복
    """
    def __init__(self, vocab_size, d_model, num_layers, num_heads, d_ff, max_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff) for _ in range(num_layers)
        ])

    def forward(self, tgt, enc_out, tgt_mask=None, src_mask=None):
        # (1) 임베딩 + 포지셔널 인코딩
        x = self.embedding(tgt)
        x = self.pos_encoding(x)
        # (2) N개 디코더 블록 반복
        for layer in self.layers:
            x = layer(x, enc_out, tgt_mask, src_mask)
        return x

# ------------------ 트랜스포머 전체 ------------------
class Transformer(nn.Module):
    """
    전체 Transformer: Encoder + Decoder + 최종 출력층
    """
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, num_layers=6, num_heads=8, d_ff=2048, max_len=512):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, d_model, num_layers, num_heads, d_ff, max_len)
        self.decoder = Decoder(tgt_vocab_size, d_model, num_layers, num_heads, d_ff, max_len)
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)

    def make_pad_mask(self, seq, pad_idx):
        # 패딩 토큰 마스킹(=attn 계산시 무시)
        return (seq != pad_idx).unsqueeze(1).unsqueeze(2)  # (batch, 1, 1, seq_len)

    def make_subsequent_mask(self, seq_len):
        # 디코더용 마스크(=미래 토큰 참조 방지)
        mask = torch.tril(torch.ones(seq_len, seq_len)).bool()
        return mask  # (seq_len, seq_len)

    def forward(self, src, tgt, src_pad_idx, tgt_pad_idx):
        # 입력/출력 시퀀스의 패딩 마스크 생성
        src_mask = self.make_pad_mask(src, src_pad_idx)  # (batch, 1, 1, src_len)
        tgt_mask = self.make_pad_mask(tgt, tgt_pad_idx) & self.make_subsequent_mask(tgt.size(1)).to(tgt.device)
        # 인코더/디코더 진행
        enc_out = self.encoder(src, src_mask)
        dec_out = self.decoder(tgt, enc_out, tgt_mask, src_mask)
        # 마지막 선형 변환으로 다음 토큰 확률 출력
        out = self.fc_out(dec_out)
        return out

# ------------------ 사용 예시 ------------------
if __name__ == "__main__":
    # 예시 파라미터
    SRC_VOCAB_SIZE = 10000
    TGT_VOCAB_SIZE = 10000
    PAD_IDX = 0

    # 모델 생성
    model = Transformer(SRC_VOCAB_SIZE, TGT_VOCAB_SIZE, d_model=512, num_layers=2, num_heads=8, d_ff=2048, max_len=100)

    # 임의의 입력 데이터(batch, seq_len)
    src = torch.randint(1, SRC_VOCAB_SIZE, (2, 15))   # (batch, src_seq_len)
    tgt = torch.randint(1, TGT_VOCAB_SIZE, (2, 12))   # (batch, tgt_seq_len)

    # 출력: (batch, tgt_seq_len, tgt_vocab_size)
    out = model(src, tgt, PAD_IDX, PAD_IDX)
    print(out.shape)  # torch.Size([2, 12, 10000])


torch.Size([2, 12, 10000])


In [4]:
import torch.optim as optim

# === 1. 한글-영어 toy 데이터 준비 ===
src_sentences = ["나는 학생이다", "나는 학교에 간다", "책을 읽는다"]
tgt_sentences = ["i am a student", "i go to school", "i read a book"]

# === 2. 토크나이즈, 단어장 생성 ===
def build_vocab(sentences):
    vocab = {"<pad>": 0, "<sos>": 1, "<eos>": 2, "<unk>": 3}
    idx = 4
    for s in sentences:
        for w in s.split():
            if w not in vocab:
                vocab[w] = idx
                idx += 1
    return vocab

src_vocab = build_vocab(src_sentences)
tgt_vocab = build_vocab(tgt_sentences)
src_ivocab = {v:k for k,v in src_vocab.items()}
tgt_ivocab = {v:k for k,v in tgt_vocab.items()}

def encode(s, vocab, maxlen):
    ids = [vocab.get(w, vocab["<unk>"]) for w in s.split()]
    ids = [vocab["<sos>"]] + ids + [vocab["<eos>"]]
    ids += [vocab["<pad>"]] * (maxlen - len(ids))
    return ids[:maxlen]

SRC_MAXLEN = 7
TGT_MAXLEN = 8
X = torch.tensor([encode(s, src_vocab, SRC_MAXLEN) for s in src_sentences])
Y = torch.tensor([encode(s, tgt_vocab, TGT_MAXLEN) for s in tgt_sentences])

# === 3. 위에 만든 Transformer 모델 그대로 불러오기 ===
SRC_VOCAB_SIZE = len(src_vocab)
TGT_VOCAB_SIZE = len(tgt_vocab)
PAD_IDX = 0

model = Transformer(SRC_VOCAB_SIZE, TGT_VOCAB_SIZE, d_model=64, num_layers=2, num_heads=4, d_ff=128, max_len=10)
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

# === 4. 모델 학습 ===
X, Y = X, Y
for epoch in range(300):
    model.train()
    optimizer.zero_grad()
    out = model(X, Y[:, :-1], PAD_IDX, PAD_IDX)  # 디코더 입력: <sos>부터 마지막-1까지
    loss = criterion(out.reshape(-1, out.size(-1)), Y[:, 1:].reshape(-1))  # 타겟: 첫번째 제외 나머지
    loss.backward()
    optimizer.step()
    if epoch % 50 == 0:
        print(f"epoch {epoch} loss {loss.item():.4f}")

# === 5. 번역(추론) 함수 ===
def translate(model, src_sentence, src_vocab, tgt_vocab, tgt_ivocab, src_maxlen, tgt_maxlen):
    model.eval()
    src_ids = torch.tensor([encode(src_sentence, src_vocab, src_maxlen)])
    ys = torch.tensor([[tgt_vocab["<sos>"]]])
    for _ in range(tgt_maxlen):
        with torch.no_grad():
            out = model(src_ids, ys, PAD_IDX, PAD_IDX)
        next_token = out[0, -1].argmax().item()
        ys = torch.cat([ys, torch.tensor([[next_token]])], dim=1)
        if next_token == tgt_vocab["<eos>"]:
            break
    # 디코딩
    result = [tgt_ivocab.get(i, "<unk>") for i in ys[0][1:-1].tolist()]
    return " ".join(result)

# === 6. 실제 번역 실행 ===
print("\n=== 실제 번역 결과 ===")
for test in ["나는 학생이다", "책을 읽는다", "나는 학교에 간다"]:
    print(f"입력: {test} → 번역: {translate(model, test, src_vocab, tgt_vocab, tgt_ivocab, SRC_MAXLEN, TGT_MAXLEN)}")


epoch 0 loss 2.4837
epoch 50 loss 0.0024
epoch 100 loss 0.0003
epoch 150 loss 0.0002
epoch 200 loss 0.0002
epoch 250 loss 0.0002

=== 실제 번역 결과 ===
입력: 나는 학생이다 → 번역: i am a student
입력: 책을 읽는다 → 번역: i read a book
입력: 나는 학교에 간다 → 번역: i go to school
